# Reusable Template: Multi-Feature Linear Regression with Scaling & Learning-Rate Diagnostics

Copy this notebook for any new tabular regression problem that needs:
- vectorized multi-variable linear regression
- learning-rate diagnosis
- feature scaling (z-score / min-max)
- out-of-sample prediction with stored transform statistics


## 1. Configuration


In [ ]:
DATA_PATH   = "data/your_data.csv"          # change
FEATURE_COLS = ["feat1", "feat2", "feat3"]  # change
TARGET_COL   = "target"                     # change
ALPHA_RAW_TRIALS = [1e-6, 3e-7, 1e-7]       # adjust to your scales
ALPHA_SCALED     = 0.1
N_ITERS_SCALED   = 1000
RANDOM_SEED      = 42


## 2. Imports & load


In [ ]:
import copy, numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(RANDOM_SEED)
df = pd.read_csv(DATA_PATH)
X = df[FEATURE_COLS].values.astype(float)
y = df[TARGET_COL].values.astype(float)
print(X.shape, y.shape)


## 3. Core routines (drop-in)


In [ ]:
def compute_cost(X, y, w, b):
    m = len(y)
    return np.sum((X @ w + b - y)**2) / (2*m)

def compute_gradient(X, y, w, b):
    m = len(y)
    err = X @ w + b - y
    return err.sum()/m, (X.T @ err)/m

def gradient_descent(X, y, w, b, alpha, n_iters):
    hist = []
    for i in range(n_iters):
        db, dw = compute_gradient(X, y, w, b)
        w, b = w - alpha*dw, b - alpha*db
        hist.append(compute_cost(X, y, w, b))
    return w, b, hist

def zscore_normalize(X):
    mu, sigma = X.mean(0), X.std(0)
    sigma = np.where(sigma==0, 1, sigma)
    return (X-mu)/sigma, mu, sigma


## 4. Diagnostic run on raw features


In [ ]:
for a in ALPHA_RAW_TRIALS:
    _,_,h = gradient_descent(X, y, np.zeros(X.shape[1]), 0., a, 40)
    print(f"α={a}: final cost={h[-1]:.4f}  (rising? {h[-1]>h[0]})")


## 5. Scale → train → predict


In [ ]:
X_n, mu, sigma = zscore_normalize(X)
w, b, hist = gradient_descent(X_n, y, np.zeros(X.shape[1]), 0., ALPHA_SCALED, N_ITERS_SCALED)
print("Final cost:", hist[-1])
print("w:", w, "b:", b)

# new example — MUST use same mu, sigma
x_new = np.array([0., 0., 0.])          # <-- replace with real values
x_new_n = (x_new - mu) / sigma
print("prediction:", x_new_n @ w + b)


## 6. Quick plots


In [ ]:
fig, ax = plt.subplots(1,2,figsize=(10,3))
ax[0].plot(hist); ax[0].set_title("Cost (scaled)")
ax[1].scatter(y, X_n@w+b, alpha=0.6)
ax[1].plot([y.min(),y.max()],[y.min(),y.max()],"r--")
ax[1].set_title("Pred vs True")
plt.tight_layout(); plt.show()
